# Nifty 50 Signal — Training v4 (India VIX + Bank Nifty + Rolling stats + SHAP + Optuna)

**v3 outcome:** 46% directional precision on holdout. Loses to mean-reversion baseline.

**v4 changes:**
1. **India VIX** features (level, regime, change) — fear gauge, single biggest external signal
2. **Bank Nifty** features (return correlation, divergence) — BNF often leads Nifty
3. **Rolling stats** (5d/10d/20d vol, mean, high-low range) — captures regime shifts
4. **SHAP feature selection** — drops low-importance features (denoises the model)
5. **Optuna hyperparameter tuning** (50 trials, ~15 min) — finds the actual best XGB params
6. **Same walk-forward CV + baselines + 90-day holdout** from v3 (honesty framework)

**Realistic target: 51-55% directional precision** — meaningful improvement, deployable for paper trading.

In [ ]:
!pip install -q ta xgboost lightgbm catboost scikit-learn shap optuna

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
DATA_DIR = '/content/drive/MyDrive/nifty_signal/data'
MODELS_DIR = '/content/drive/MyDrive/nifty_signal/models'
os.makedirs(MODELS_DIR, exist_ok=True)

all_files = os.listdir(DATA_DIR)
print('Files in DATA_DIR:')
for f in sorted(all_files):
    print(f'  {f}')

def find_file(patterns, files):
    for pat in patterns:
        for f in files:
            if re.search(pat, f, re.IGNORECASE):
                return f
    return None

NIFTY_DAILY  = find_file([r'nifty.*50.*daily', r'^nifty50_daily', r'^nifty.*daily'], all_files)
NIFTY_HOURLY = find_file([r'nifty.*50.*hourly', r'^nifty50_hourly', r'^nifty.*hour'], all_files)
VIX_FILE     = find_file([r'india.*vix', r'^vix', r'_vix\.csv'], all_files)
BNF_FILE     = find_file([r'bank.*nifty', r'nifty.*bank'], all_files)

print(f'\nDetected:')
print(f'  Nifty daily:  {NIFTY_DAILY}')
print(f'  Nifty hourly: {NIFTY_HOURLY}')
print(f'  India VIX:    {VIX_FILE}')
print(f'  Bank Nifty:   {BNF_FILE}')

missing = [n for n, f in [('Nifty daily', NIFTY_DAILY), ('VIX', VIX_FILE), ('BankNifty', BNF_FILE)] if not f]
if missing:
    print(f'\n⚠️  Missing: {missing}. Upload them to {DATA_DIR} before continuing.')
else:
    print('\nAll required files present ✓')

In [ ]:
import pandas as pd, numpy as np
from ta.momentum import RSIIndicator
from ta.trend import MACD, EMAIndicator, ADXIndicator
from ta.volatility import AverageTrueRange, BollingerBands

def load_csv(path):
    df = pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    for c in ('date','timestamp','time'):
        if c in df.columns and 'datetime' not in df.columns:
            df = df.rename(columns={c:'datetime'})
            break
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['date'] = df['datetime'].dt.tz_localize(None).dt.normalize() if df['datetime'].dt.tz else df['datetime'].dt.normalize()
    return df.sort_values('datetime').reset_index(drop=True)

# ---------- Feature lists (will be defined as features are added) ----------
FEATURES_NIFTY = []     # core Nifty technicals + lags + rolling stats
FEATURES_INTRADAY = []  # from hourly aggregation
FEATURES_VIX = []       # India VIX
FEATURES_BNF = []       # Bank Nifty

In [ ]:
def add_nifty_features(df):
    out = df.copy()
    c, h, l, v = out['close'], out['high'], out['low'], out['volume']

    # ----- Classical technicals -----
    out['rsi_14']   = RSIIndicator(c, 14).rsi()
    out['macd_hist']= MACD(c, 26, 12, 9).macd_diff()
    ema9, ema21, ema50 = EMAIndicator(c, 9).ema_indicator(), EMAIndicator(c, 21).ema_indicator(), EMAIndicator(c, 50).ema_indicator()
    out['ema_cross_9_21']  = (ema9 > ema21).astype(int)
    out['ema_cross_21_50'] = (ema21 > ema50).astype(int)
    out['atr_14'] = AverageTrueRange(h, l, c, 14).average_true_range()
    bb = BollingerBands(c, 20, 2)
    out['bb_pct_b'] = (c - bb.bollinger_lband()) / (bb.bollinger_hband() - bb.bollinger_lband())
    out['adx_14']   = ADXIndicator(h, l, c, 14).adx()

    # ----- Lagged returns -----
    for lag in (1, 3, 5, 10, 20):
        out[f'ret_{lag}d'] = c.pct_change(lag)

    # ----- NEW v4: Rolling stats -----
    rets = c.pct_change()
    for w in (5, 10, 20):
        out[f'vol_{w}d']  = rets.rolling(w).std()
        out[f'mean_{w}d'] = rets.rolling(w).mean()
        out[f'high_{w}d_dist'] = (c - h.rolling(w).max()) / h.rolling(w).max()
        out[f'low_{w}d_dist']  = (c - l.rolling(w).min()) / l.rolling(w).min()

    # Volatility regime
    atr_50 = AverageTrueRange(h, l, c, 50).average_true_range()
    out['vol_regime'] = out['atr_14'] / atr_50
    out['atr_pct']    = out['atr_14'] / c

    # Volume features (robust to zero-volume index data)
    v_safe = v.replace(0, np.nan)
    out['log_vol']     = np.log1p(v_safe).fillna(0)
    vma20, vstd20 = v_safe.rolling(20).mean(), v_safe.rolling(20).std()
    out['vol_vs_ma20'] = (v_safe / vma20).fillna(1.0)
    out['vol_zscore']  = ((v_safe - vma20) / vstd20).fillna(0.0)

    # Gap
    out['gap_pct'] = (out['open'] - c.shift(1)) / c.shift(1)

    # RSI divergence
    out['rsi_divergence'] = (np.sign(c.pct_change(14)) * np.sign(out['rsi_14'].diff(14))).fillna(0)

    # Calendar
    dt = pd.to_datetime(out['datetime'])
    out['day_of_week'] = dt.dt.dayofweek
    out['dte']         = (3 - dt.dt.dayofweek) % 7

    return out

FEATURES_NIFTY = [
    # Classical
    'rsi_14','macd_hist','ema_cross_9_21','ema_cross_21_50','atr_14','bb_pct_b','adx_14',
    # Lagged returns
    'ret_1d','ret_3d','ret_5d','ret_10d','ret_20d',
    # Rolling stats (NEW in v4)
    'vol_5d','vol_10d','vol_20d',
    'mean_5d','mean_10d','mean_20d',
    'high_5d_dist','high_10d_dist','high_20d_dist',
    'low_5d_dist','low_10d_dist','low_20d_dist',
    # Regime / vol
    'vol_regime','atr_pct',
    # Volume
    'log_vol','vol_vs_ma20','vol_zscore',
    # Other
    'gap_pct','rsi_divergence',
    'day_of_week','dte',
]
print(f'{len(FEATURES_NIFTY)} Nifty features defined')

In [ ]:
def hourly_to_daily_features(hourly_df):
    h = hourly_df.copy()
    h['intraday_ret'] = (h['close'] - h['open']) / h['open']
    grouped = h.groupby('date').agg(
        intraday_total_return=('close', lambda s: (s.iloc[-1] - s.iloc[0]) / s.iloc[0] if len(s) > 1 else 0),
        intraday_max_drawdown=('close', lambda s: (s.min() - s.iloc[0]) / s.iloc[0] if len(s) > 1 else 0),
        intraday_vol=('intraday_ret', 'std'),
        intraday_volume_total=('volume', 'sum'),
        intraday_n_bars=('close', 'count'),
    ).reset_index()
    grouped['intraday_volume_total'] = np.log1p(grouped['intraday_volume_total'].fillna(0))
    grouped['intraday_vol'] = grouped['intraday_vol'].fillna(0)
    return grouped

FEATURES_INTRADAY = ['intraday_total_return','intraday_max_drawdown','intraday_vol','intraday_volume_total','intraday_n_bars']
print(f'{len(FEATURES_INTRADAY)} intraday-aggregate features')

In [ ]:
def build_vix_features(vix_df):
    """India VIX features. Expects columns: date, close (and ideally open/high/low)."""
    v = vix_df.copy()
    # Some VIX CSVs use only 'value' or 'close' — handle both
    vix_col = 'close' if 'close' in v.columns else ('value' if 'value' in v.columns else None)
    if vix_col is None:
        raise ValueError(f'VIX CSV needs close or value column. Got: {list(v.columns)}')
    
    v = v[['date', vix_col]].rename(columns={vix_col: 'vix'})
    v['vix_change_1d'] = v['vix'].pct_change(1)
    v['vix_change_5d'] = v['vix'].pct_change(5)
    v['vix_ma20']      = v['vix'].rolling(20).mean()
    v['vix_vs_ma20']   = v['vix'] / v['vix_ma20']
    # Regime flags: high vol (VIX>20), low vol (VIX<14)
    v['vix_high']      = (v['vix'] > 20).astype(int)
    v['vix_low']       = (v['vix'] < 14).astype(int)
    return v.drop(columns=['vix_ma20'])

FEATURES_VIX = ['vix','vix_change_1d','vix_change_5d','vix_vs_ma20','vix_high','vix_low']
print(f'{len(FEATURES_VIX)} VIX features')

In [ ]:
def build_bnf_features(bnf_df, nifty_close: pd.Series):
    """Bank Nifty features merged on date with cross-asset signals."""
    b = bnf_df.copy()
    b = b[['date','close','high','low']].rename(columns={'close':'bnf_close','high':'bnf_high','low':'bnf_low'})
    b['bnf_ret_1d']  = b['bnf_close'].pct_change(1)
    b['bnf_ret_5d']  = b['bnf_close'].pct_change(5)
    b['bnf_ret_20d'] = b['bnf_close'].pct_change(20)
    bnf_rets = b['bnf_close'].pct_change()
    nifty_rets = nifty_close.pct_change()
    # 20-day rolling correlation between Nifty and Bank Nifty
    b['bnf_nifty_corr_20'] = bnf_rets.rolling(20).corr(nifty_rets)
    # Divergence: BNF moving up while Nifty down (or vice versa)
    b['bnf_nifty_divergence'] = np.sign(b['bnf_ret_1d']) - np.sign(nifty_rets)
    return b.drop(columns=['bnf_high','bnf_low'])

FEATURES_BNF = ['bnf_ret_1d','bnf_ret_5d','bnf_ret_20d','bnf_nifty_corr_20','bnf_nifty_divergence']
print(f'{len(FEATURES_BNF)} Bank Nifty features')

## Build the full dataset

In [ ]:
# Load Nifty daily + features
nifty = load_csv(f'{DATA_DIR}/{NIFTY_DAILY}')
feats = add_nifty_features(nifty)
print(f'[1] Nifty: {len(feats)} rows, range {feats["date"].min().date()} to {feats["date"].max().date()}')

# Merge intraday (hourly aggregates)
if NIFTY_HOURLY:
    hourly = load_csv(f'{DATA_DIR}/{NIFTY_HOURLY}')
    intraday = hourly_to_daily_features(hourly)
    feats = feats.merge(intraday, on='date', how='left')
    for col in FEATURES_INTRADAY:
        feats[col] = feats[col].fillna(0)
    print(f'[2] Merged intraday: {feats["intraday_total_return"].notna().sum()} day-matches')
else:
    for col in FEATURES_INTRADAY:
        feats[col] = 0
    print('[2] No hourly file — intraday features zeroed')

# Merge VIX
vix_raw = load_csv(f'{DATA_DIR}/{VIX_FILE}')
vix = build_vix_features(vix_raw)
feats = feats.merge(vix, on='date', how='left')
for col in FEATURES_VIX:
    feats[col] = feats[col].ffill().bfill()  # carry VIX across small gaps
print(f'[3] Merged VIX: range {vix["date"].min().date()} to {vix["date"].max().date()}')

# Merge Bank Nifty
bnf_raw = load_csv(f'{DATA_DIR}/{BNF_FILE}')
bnf = build_bnf_features(bnf_raw, nifty.set_index('date')['close'])
# bnf is indexed by date — re-set as column-form
if 'date' not in bnf.columns:
    bnf = bnf.reset_index()
feats = feats.merge(bnf[['date'] + FEATURES_BNF], on='date', how='left')
for col in FEATURES_BNF:
    feats[col] = feats[col].ffill().fillna(0)
print(f'[4] Merged Bank Nifty: range {bnf_raw["date"].min().date()} to {bnf_raw["date"].max().date()}')

ALL_FEATURES = FEATURES_NIFTY + FEATURES_INTRADAY + FEATURES_VIX + FEATURES_BNF
print(f'\n=== Total features: {len(ALL_FEATURES)} ===')
print(f'  Nifty:    {len(FEATURES_NIFTY)}')
print(f'  Intraday: {len(FEATURES_INTRADAY)}')
print(f'  VIX:      {len(FEATURES_VIX)}')
print(f'  BankNFTY: {len(FEATURES_BNF)}')

# Add target + sanitize
feats['next_return'] = feats['close'].pct_change().shift(-1)
feats[ALL_FEATURES] = feats[ALL_FEATURES].replace([np.inf, -np.inf], np.nan)
feats = feats.dropna(subset=['next_return']).reset_index(drop=True)
n_before = len(feats)
feats = feats.dropna(subset=ALL_FEATURES).reset_index(drop=True)
print(f'\nAfter dropna: {len(feats)} rows (lost {n_before - len(feats)})')

# Clip outliers
for col in ALL_FEATURES:
    if feats[col].dtype.kind in 'fc':
        q01, q99 = feats[col].quantile([0.001, 0.999])
        feats[col] = feats[col].clip(q01, q99)

print(f'\nFinal dataset: {len(feats)} rows, {feats["date"].min().date()} to {feats["date"].max().date()}')

In [ ]:
# Holdout split
HOLDOUT_DAYS = 90
cutoff = feats['date'].max() - pd.Timedelta(days=HOLDOUT_DAYS)
df_train = feats[feats['date'] <= cutoff].reset_index(drop=True)
df_holdout = feats[feats['date'] > cutoff].reset_index(drop=True)
print(f'Training pool: {len(df_train)} rows ({df_train["date"].min().date()} to {df_train["date"].max().date()})')
print(f'Holdout test:  {len(df_holdout)} rows ({df_holdout["date"].min().date()} to {df_holdout["date"].max().date()})')

def make_labels(next_returns, q_low=0.33, q_high=0.67):
    lo, hi = next_returns.quantile(q_low), next_returns.quantile(q_high)
    lab = pd.Series(1, index=next_returns.index)
    lab[next_returns > hi] = 2
    lab[next_returns < lo] = 0
    return lab, lo, hi

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
import warnings; warnings.filterwarnings('ignore')

X_pool = df_train[ALL_FEATURES].values
ret_pool = df_train['next_return'].values
prev_ret_pool = df_train['ret_1d'].fillna(0).values

def directional_precision(y_true, y_pred):
    mask = y_pred != 1
    if mask.sum() == 0:
        return 0.0
    return (y_pred[mask] == y_true[mask]).mean()

def cv_score(X, ret_arr, features_subset_idx=None, n_splits=5, params=None):
    if features_subset_idx is not None:
        X = X[:, features_subset_idx]
    p = dict(n_estimators=500, max_depth=5, learning_rate=0.04,
             objective='multi:softprob', num_class=3, eval_metric='mlogloss',
             tree_method='hist', device='cuda',
             reg_alpha=0.1, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8)
    if params: p.update(params)
    
    tscv = TimeSeriesSplit(n_splits=n_splits)
    scores = []
    for tr, te in tscv.split(X):
        train_ret = pd.Series(ret_arr[tr])
        _, lo, hi = make_labels(train_ret)
        labels = pd.Series(1, index=range(len(ret_arr)))
        labels[pd.Series(ret_arr) > hi] = 2
        labels[pd.Series(ret_arr) < lo] = 0
        y_tr, y_te = labels.iloc[tr].values, labels.iloc[te].values
        sw = compute_sample_weight('balanced', y_tr)
        m = XGBClassifier(**p)
        m.fit(X[tr], y_tr, sample_weight=sw)
        scores.append(directional_precision(y_te, m.predict(X[te])))
    return np.mean(scores), np.std(scores)

print('Baseline CV with all v4 features:')
mean_dp, std_dp = cv_score(X_pool, ret_pool)
print(f'  Mean directional precision: {mean_dp:.3f}  (std {std_dp:.3f})')

## SHAP analysis — find which features actually matter

In [ ]:
import shap

# Train one XGB on the most recent 80% of training pool to get SHAP importances
split = int(len(X_pool) * 0.8)
X_tr, X_val = X_pool[:split], X_pool[split:]
ret_tr = pd.Series(ret_pool[:split])
_, lo, hi = make_labels(ret_tr)
y_tr = pd.Series(1, index=range(split))
y_tr[ret_tr > hi] = 2; y_tr[ret_tr < lo] = 0; y_tr = y_tr.values
y_val = pd.Series(1, index=range(len(X_val)))
ret_val = pd.Series(ret_pool[split:])
y_val[ret_val > hi] = 2; y_val[ret_val < lo] = 0; y_val = y_val.values

sw = compute_sample_weight('balanced', y_tr)
shap_model = XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.04,
                            objective='multi:softprob', num_class=3, eval_metric='mlogloss',
                            tree_method='hist', device='cuda',
                            reg_alpha=0.1, reg_lambda=1.0, subsample=0.8, colsample_bytree=0.8)
shap_model.fit(X_tr, y_tr, sample_weight=sw)

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_val)

# Handle all SHAP return shapes:
#   - list of arrays [n_classes] each (n_samples, n_features) — older shap
#   - 3D ndarray (n_samples, n_features, n_classes) — newer shap
#   - 2D ndarray (n_samples, n_features) — binary case
if isinstance(shap_values, list):
    mean_abs = np.mean([np.abs(arr).mean(axis=0) for arr in shap_values], axis=0)
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    # (samples, features, classes) -> mean abs across samples, then across classes
    mean_abs = np.abs(shap_values).mean(axis=0).mean(axis=-1)
else:
    mean_abs = np.abs(shap_values).mean(axis=0)

# Sanity check the shape
mean_abs = np.asarray(mean_abs).reshape(-1)
assert len(mean_abs) == len(ALL_FEATURES), f'SHAP shape mismatch: {mean_abs.shape} vs {len(ALL_FEATURES)} features'

importance = pd.DataFrame({'feature': ALL_FEATURES, 'shap': mean_abs}).sort_values('shap', ascending=False)
print('Top 20 features by SHAP importance:')
print(importance.head(20).to_string(index=False))
print('\nBottom 10 (candidates to drop):')
print(importance.tail(10).to_string(index=False))

In [ ]:
# Keep top-K features (drop the noisiest ~30%)
K = max(20, int(len(ALL_FEATURES) * 0.7))
TOP_FEATURES = importance.head(K)['feature'].tolist()
TOP_IDX = [ALL_FEATURES.index(f) for f in TOP_FEATURES]
print(f'Kept top {K} of {len(ALL_FEATURES)} features')

mean_dp_after, std_dp_after = cv_score(X_pool, ret_pool, features_subset_idx=TOP_IDX)
print(f'\nCV with top-{K} features: mean dir_prec = {mean_dp_after:.3f}  (std {std_dp_after:.3f})')
print(f'CV with all {len(ALL_FEATURES)} features: {mean_dp:.3f}  → delta {mean_dp_after - mean_dp:+.3f}')

## Optuna hyperparameter tuning

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_top = X_pool[:, TOP_IDX]

def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda':       trial.suggest_float('reg_lambda', 0.5, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    mean_dp, _ = cv_score(X_top, ret_pool, n_splits=3, params=params)
    return mean_dp

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest dir_prec: {study.best_value:.3f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')
BEST_PARAMS = study.best_params

## Final training + out-of-time holdout

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib

X_train_final = df_train[TOP_FEATURES].values
X_hold = df_holdout[TOP_FEATURES].values
ret_full = df_train['next_return']
_, lo, hi = make_labels(ret_full)
y_full = pd.Series(1, index=ret_full.index); y_full[ret_full > hi] = 2; y_full[ret_full < lo] = 0
y_full = y_full.values
ret_hold = df_holdout['next_return']
y_hold = pd.Series(1, index=ret_hold.index); y_hold[ret_hold > hi] = 2; y_hold[ret_hold < lo] = 0
y_hold = y_hold.values
sw_full = compute_sample_weight('balanced', y_full)

xgb_params = dict(
    objective='multi:softprob', num_class=3, eval_metric='mlogloss',
    tree_method='hist', device='cuda', **BEST_PARAMS
)
xgb_final  = XGBClassifier(**xgb_params)
lgbm_final = LGBMClassifier(n_estimators=BEST_PARAMS['n_estimators'], max_depth=BEST_PARAMS['max_depth'],
                            learning_rate=BEST_PARAMS['learning_rate'],
                            subsample=BEST_PARAMS['subsample'], colsample_bytree=BEST_PARAMS['colsample_bytree'],
                            reg_alpha=BEST_PARAMS['reg_alpha'], reg_lambda=BEST_PARAMS['reg_lambda'],
                            objective='multiclass', num_class=3, class_weight='balanced', verbose=-1)
cat_final  = CatBoostClassifier(iterations=BEST_PARAMS['n_estimators'], depth=BEST_PARAMS['max_depth'],
                                 learning_rate=BEST_PARAMS['learning_rate'],
                                 loss_function='MultiClass', auto_class_weights='Balanced',
                                 task_type='GPU', verbose=0,
                                 l2_leaf_reg=BEST_PARAMS['reg_lambda'])

xgb_final.fit(X_train_final, y_full, sample_weight=sw_full)
lgbm_final.fit(X_train_final, y_full)
cat_final.fit(X_train_final, y_full)

# Stacking meta on held-out 20% of train
split = int(len(X_train_final) * 0.8)
Xm_tr, Xm_val = X_train_final[:split], X_train_final[split:]
ym_tr, ym_val = y_full[:split], y_full[split:]
swm = compute_sample_weight('balanced', ym_tr)
xgb_m = XGBClassifier(**xgb_params);  xgb_m.fit(Xm_tr, ym_tr, sample_weight=swm)
lgbm_m = LGBMClassifier(n_estimators=BEST_PARAMS['n_estimators'], objective='multiclass', num_class=3, class_weight='balanced', verbose=-1); lgbm_m.fit(Xm_tr, ym_tr)
cat_m  = CatBoostClassifier(iterations=BEST_PARAMS['n_estimators'], loss_function='MultiClass', auto_class_weights='Balanced', task_type='GPU', verbose=0); cat_m.fit(Xm_tr, ym_tr)
base_val = np.hstack([xgb_m.predict_proba(Xm_val), lgbm_m.predict_proba(Xm_val), cat_m.predict_proba(Xm_val)])
meta = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.3)
meta.fit(base_val, ym_val)
cal_meta = CalibratedClassifierCV(meta, method='sigmoid', cv='prefit'); cal_meta.fit(base_val, ym_val)

# Holdout
base_h = np.hstack([xgb_final.predict_proba(X_hold), lgbm_final.predict_proba(X_hold), cat_final.predict_proba(X_hold)])
y_pred = cal_meta.predict(base_h)
y_proba = cal_meta.predict_proba(base_h)
acc = accuracy_score(y_hold, y_pred)
dp = directional_precision(y_hold, y_pred)

print('=== v4 FINAL HOLDOUT ===')
print(f'MODEL: acc={acc:.3f}  dir_prec={dp:.3f}  ({(y_pred!=1).sum()} directional preds)')

# Baselines for comparison
prev_hold = df_holdout['ret_1d'].fillna(0).values
from sklearn.metrics import accuracy_score as accs
for name, preds in [
    ('Always BUY_CALL', np.full(len(y_hold), 2)),
    ('Always BUY_PUT',  np.full(len(y_hold), 0)),
    ('Momentum',        np.where(prev_hold > 0, 2, np.where(prev_hold < 0, 0, 1))),
    ('Mean-Reversion',  np.where(prev_hold > 0, 0, np.where(prev_hold < 0, 2, 1))),
]:
    bdp = directional_precision(y_hold, preds)
    print(f'  {name:18s}: acc={accs(y_hold, preds):.3f}  dir_prec={bdp:.3f}')

print('\nConfidence-gated holdout precision:')
maxp = y_proba.max(axis=1)
for t in (0.0, 0.4, 0.5, 0.6, 0.7, 0.8):
    m = (maxp >= t) & (y_pred != 1)
    if m.sum() == 0:
        print(f'  conf>={t:.1f}: no trades')
        continue
    p = (y_pred[m] == y_hold[m]).mean()
    print(f'  conf>={t:.1f}: n={m.sum():3d}  precision={p:.3f}')

In [ ]:
# Save everything
import json
joblib.dump(xgb_final, f'{MODELS_DIR}/xgboost.pkl')
joblib.dump(lgbm_final, f'{MODELS_DIR}/lightgbm.pkl')
joblib.dump(cat_final, f'{MODELS_DIR}/catboost.pkl')
joblib.dump(cal_meta, f'{MODELS_DIR}/meta_rf.pkl')

with open(f'{MODELS_DIR}/label_thresholds.json', 'w') as f:
    json.dump({
        'q_low_return': float(lo), 'q_high_return': float(hi),
        'features': TOP_FEATURES,
        'all_features_considered': ALL_FEATURES,
        'best_params': BEST_PARAMS,
        'holdout_dir_prec': float(dp),
    }, f, indent=2)

print(f'\nSaved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    sz = os.path.getsize(f'{MODELS_DIR}/{f}') / 1e6
    print(f'  {f}  ({sz:.2f} MB)')

## Decision guide

Compare v4 holdout `dir_prec` to v3's 46% baseline:

- **dir_prec > 0.55** AND **rises with confidence** → ship it. Update backend's `FEATURE_COLUMNS` to match `TOP_FEATURES`.
- **dir_prec 0.50-0.55** → marginal improvement. Deploy for paper trading, watch live performance.
- **dir_prec < 0.50** → adding VIX/BNF/rolling stats didn't help. Next move is Tier 3 — predict big moves only, or pivot to volatility prediction.

If shipping, you must also update `backend/feature_engineering.py` to produce all `TOP_FEATURES`. Tell me the holdout result and I'll patch the backend.